# 00_1. Repair and Materialize BDD100K for FedSTO

Use this notebook when `00_download_and_inspect_bdd100k.ipynb` succeeded enough to create metadata, but training later reports corrupt images, missing labels, or stale EfficientTeacher cache files.

What this notebook does:
- finds the current project root without hard-coded machine paths,
- finds or downloads the KaggleHub BDD100K mirror,
- verifies required raw images with PIL,
- rebuilds `data_paper20k/` with real copied image files instead of symlinks,
- removes stale `.cache` files,
- regenerates EfficientTeacher data lists/configs,
- verifies the rebuilt copied images before running `03`.

This is intentionally heavier than symlinks, but it is safer for Docker, copied workspaces, and remote machines.


## 1. Setup


In [1]:
from __future__ import annotations

import csv
import json
import os
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path
from typing import Optional

import yaml
from PIL import Image


## 2. Locate Project and Raw BDD100K


In [2]:
def find_project_root(start: Optional[Path] = None) -> Path:
    start = Path.cwd().resolve() if start is None else Path(start).resolve()
    markers = (
        "prepare_bdd100k_for_fedsto.py",
        "prepare_bdd10k_for_fedsto.py",
        "00_download_and_inspect_bdd100k.ipynb",
        "00_1_repair_and_materialize_bdd100k.ipynb",
    )
    candidate_dirs = []
    for base in (start, *start.parents):
        candidate_dirs.extend([
            base,
            base / "navigating_data_heterogeneity",
            base / "Object_Detection" / "navigating_data_heterogeneity",
            base / "masters_research" / "navigating_data_heterogeneity",
        ])
    for candidate in candidate_dirs:
        if any((candidate / marker).exists() for marker in markers):
            return candidate.resolve()
    for pattern in (
        "*/prepare_bdd100k_for_fedsto.py",
        "*/*/prepare_bdd100k_for_fedsto.py",
        "*/prepare_bdd10k_for_fedsto.py",
        "*/*/prepare_bdd10k_for_fedsto.py",
    ):
        matches = list(start.glob(pattern))
        if matches:
            return matches[0].parent.resolve()
    return start


PROJECT_ROOT = find_project_root()
RAW_ROOT = PROJECT_ROOT / "raw" / "bdd100k"
REPAIR_REPORT_ROOT = PROJECT_ROOT / "reports" / "bdd100k_repair"
PAPER_DATA_ROOT = PROJECT_ROOT / "data_paper20k"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("RAW_ROOT:", RAW_ROOT)
print("PAPER_DATA_ROOT:", PAPER_DATA_ROOT)


PROJECT_ROOT: /app/Object_Detection/navigating_data_heterogeneity
RAW_ROOT: /app/Object_Detection/navigating_data_heterogeneity/raw/bdd100k
PAPER_DATA_ROOT: /app/Object_Detection/navigating_data_heterogeneity/data_paper20k


In [3]:
DATASET_HANDLE = "awsaf49/bdd100k-dataset"

# Set this to True only when the recorded KaggleHub cache itself is broken and you want to download again.
FORCE_KAGGLEHUB_DOWNLOAD = False

# If FORCE_KAGGLEHUB_DOWNLOAD=True and this is also True, the recorded KaggleHub dataset directory is deleted first.
# Leave False unless you are sure the recorded cache is the broken one.
PURGE_RECORDED_KAGGLE_ROOT = False


def bdd_layout_ok(root: Path) -> bool:
    root = Path(root)
    return (
        (root / "labels" / "det_v2_train_release.json").exists()
        and (root / "labels" / "det_v2_val_release.json").exists()
        and (root / "bdd100k" / "bdd100k" / "images" / "100k" / "train").exists()
        and (root / "bdd100k" / "bdd100k" / "images" / "100k" / "val").exists()
    )


def candidate_raw_roots() -> list[Path]:
    candidates = []
    env_root = os.environ.get("BDD100K_ROOT")
    if env_root:
        candidates.append(Path(env_root))
    path_file = RAW_ROOT / "kagglehub_path.txt"
    if path_file.exists():
        candidates.append(Path(path_file.read_text(encoding="utf-8").strip()))
    candidates.extend([
        RAW_ROOT / "downloaded",
        Path.home() / ".cache" / "kagglehub" / "datasets" / "awsaf49" / "bdd100k-dataset" / "versions" / "1",
        Path("/root/.cache/kagglehub/datasets/awsaf49/bdd100k-dataset/versions/1"),
        Path("/app/.cache/kagglehub/datasets/awsaf49/bdd100k-dataset/versions/1"),
    ])
    unique = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.expanduser().resolve()
        except FileNotFoundError:
            resolved = candidate.expanduser()
        if str(resolved) not in seen:
            unique.append(resolved)
            seen.add(str(resolved))
    return unique


def download_with_kagglehub() -> Path:
    try:
        import kagglehub
    except ImportError as exc:
        raise ImportError(
            f"kagglehub is required. Install with: {sys.executable} -m pip install kagglehub"
        ) from exc

    recorded = RAW_ROOT / "kagglehub_path.txt"
    if PURGE_RECORDED_KAGGLE_ROOT and recorded.exists():
        old_root = Path(recorded.read_text(encoding="utf-8").strip())
        if old_root.exists() and ".cache/kagglehub" in str(old_root):
            print("Removing recorded KaggleHub root:", old_root)
            shutil.rmtree(old_root)

    path = Path(kagglehub.dataset_download(DATASET_HANDLE)).resolve()
    RAW_ROOT.mkdir(parents=True, exist_ok=True)
    recorded.write_text(str(path), encoding="utf-8")
    link_path = RAW_ROOT / "downloaded"
    try:
        if link_path.exists() or link_path.is_symlink():
            link_path.unlink()
        link_path.symlink_to(path, target_is_directory=True)
    except OSError as exc:
        print("Symlink skipped:", exc)
    return path


raw_root = None
if not FORCE_KAGGLEHUB_DOWNLOAD:
    for candidate in candidate_raw_roots():
        if bdd_layout_ok(candidate):
            raw_root = candidate
            break

if raw_root is None:
    raw_root = download_with_kagglehub()

if not bdd_layout_ok(raw_root):
    checked = "\n".join(str(path) for path in candidate_raw_roots())
    raise FileNotFoundError(f"BDD100K layout was not found. Checked:\n{checked}")

RAW_ROOT.mkdir(parents=True, exist_ok=True)
(RAW_ROOT / "kagglehub_path.txt").write_text(str(raw_root), encoding="utf-8")
print("BDD100K raw root:", raw_root)


BDD100K raw root: /root/.cache/kagglehub/datasets/awsaf49/bdd100k-dataset/versions/1


## 3. Raw Dataset Sanity Check


In [4]:
def label_json(root: Path, split: str) -> Path:
    return root / "labels" / f"det_v2_{split}_release.json"


def image_root(root: Path, split: str) -> Path:
    return root / "bdd100k" / "bdd100k" / "images" / "100k" / split


def load_records(root: Path, split: str) -> list[dict]:
    return json.loads(label_json(root, split).read_text(encoding="utf-8"))


train_records = load_records(raw_root, "train")
val_records = load_records(raw_root, "val")
train_images = list(image_root(raw_root, "train").glob("*.jpg"))
val_images = list(image_root(raw_root, "val").glob("*.jpg"))

print("train records:", len(train_records))
print("val records:", len(val_records))
print("train jpg files:", len(train_images))
print("val jpg files:", len(val_images))
print("train label json:", label_json(raw_root, "train"))
print("val label json:", label_json(raw_root, "val"))


train records: 69863
val records: 10000
train jpg files: 70000
val jpg files: 10000
train label json: /root/.cache/kagglehub/datasets/awsaf49/bdd100k-dataset/versions/1/labels/det_v2_train_release.json
val label json: /root/.cache/kagglehub/datasets/awsaf49/bdd100k-dataset/versions/1/labels/det_v2_val_release.json


## 4. Build Verified Paper20K Split With Copied Images


In [5]:
import prepare_bdd100k_for_fedsto as prep

SERVER_WEATHER = "partly cloudy"
CLIENT_WEATHERS = ("overcast", "rainy", "snowy")
MAX_SERVER_TRAIN = 5000
MAX_CLIENT_IMAGES = 5000

prep.DATA_ROOT = PAPER_DATA_ROOT
REPAIR_REPORT_ROOT.mkdir(parents=True, exist_ok=True)


def verify_image(path: Path) -> tuple[bool, tuple[int, int] | None, str]:
    if not path.exists():
        return False, None, "missing"
    try:
        with Image.open(path) as img:
            img.verify()
        with Image.open(path) as img:
            return True, img.size, "ok"
    except Exception as exc:
        return False, None, f"{type(exc).__name__}: {exc}"


def reset_dir(path: Path) -> None:
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)


def copy_image(src: Path, dst: Path) -> None:
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists() or dst.is_symlink():
        dst.unlink()
    shutil.copy2(src, dst)


def write_csv(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if not rows:
        path.write_text("", encoding="utf-8")
        return
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)


def select_verified_server(records: list[dict], split: str, weather: str, limit: int | None):
    selected = []
    bad = []
    src_root = image_root(raw_root, split)
    for record in records:
        if prep.record_attr(record, "weather") != weather:
            continue
        src = src_root / record["name"]
        ok, size, reason = verify_image(src)
        if not ok or size is None:
            bad.append({"split": split, "name": record["name"], "path": str(src), "reason": reason})
            continue
        lines = prep.mapped_yolo_lines(record, size)
        if not lines:
            continue
        selected.append((record, lines, src))
        if limit is not None and len(selected) >= limit:
            break
    return selected, bad


def select_verified_client(records: list[dict], weather: str, limit: int):
    selected = []
    bad = []
    src_root = image_root(raw_root, "train")
    for record in records:
        if prep.record_attr(record, "weather") != weather:
            continue
        src = src_root / record["name"]
        ok, _size, reason = verify_image(src)
        if not ok:
            bad.append({"split": "train", "name": record["name"], "path": str(src), "reason": reason, "weather": weather})
            continue
        selected.append((record, src))
        if len(selected) >= limit:
            break
    return selected, bad


def write_server_split(split: str, selected):
    out_img_dir = PAPER_DATA_ROOT / "server" / "images" / split
    out_label_dir = PAPER_DATA_ROOT / "server" / "labels" / split
    out_img_dir.mkdir(parents=True, exist_ok=True)
    out_label_dir.mkdir(parents=True, exist_ok=True)
    class_counts = Counter()
    box_count = 0
    for record, lines, src in selected:
        copy_image(src, out_img_dir / record["name"])
        label_path = out_label_dir / f"{Path(record['name']).stem}.txt"
        label_path.write_text("\n".join(lines) + "\n", encoding="utf-8")
        box_count += len(lines)
        for line in lines:
            class_counts[prep.YOLO_NAMES[int(line.split()[0])]] += 1
    return {"split": split, "num_images": len(selected), "num_boxes": box_count, "class_counts": dict(class_counts)}


def write_client_split(client_id: int, weather: str, selected):
    out_dir = PAPER_DATA_ROOT / "clients" / f"client_{client_id}" / "images_unlabeled"
    out_dir.mkdir(parents=True, exist_ok=True)
    for record, src in selected:
        copy_image(src, out_dir / record["name"])
    return {"client_id": client_id, "weather": weather, "num_images": len(selected)}


def write_server_yaml() -> None:
    payload = {
        "path": str((PAPER_DATA_ROOT / "server").resolve()),
        "train": "images/train",
        "val": "images/val",
        "nc": len(prep.YOLO_NAMES),
        "names": prep.YOLO_NAMES,
    }
    (PAPER_DATA_ROOT / "server").mkdir(parents=True, exist_ok=True)
    (PAPER_DATA_ROOT / "server" / "data.yaml").write_text(yaml.safe_dump(payload, sort_keys=False, allow_unicode=True), encoding="utf-8")


reset_dir(PAPER_DATA_ROOT)

server_train, bad_train = select_verified_server(train_records, "train", SERVER_WEATHER, MAX_SERVER_TRAIN)
server_val, bad_val = select_verified_server(val_records, "val", SERVER_WEATHER, None)

if not server_train:
    raise RuntimeError("No verified server train images were found. Check raw BDD100K images.")
if not server_val:
    raise RuntimeError("No verified server val images were found. Check raw BDD100K images.")

server_summary = [write_server_split("train", server_train), write_server_split("val", server_val)]
write_server_yaml()

client_summary = []
bad_clients = []
for client_id, weather in enumerate(CLIENT_WEATHERS):
    selected, bad = select_verified_client(train_records, weather, MAX_CLIENT_IMAGES)
    client_summary.append(write_client_split(client_id, weather, selected))
    bad_clients.extend(bad)

bad_rows = bad_train + bad_val + bad_clients
write_csv(REPAIR_REPORT_ROOT / "bad_required_images.csv", bad_rows)

unlabeled_total = sum(item["num_images"] for item in client_summary)
summary = {
    "server_weather": SERVER_WEATHER,
    "client_weathers": list(CLIENT_WEATHERS),
    "server": server_summary,
    "clients": client_summary,
    "data_yaml": str((PAPER_DATA_ROOT / "server" / "data.yaml").resolve()),
    "class_mapping": prep.BDD_TO_YOLO_CLASS,
    "paper_reproduction": {
        "target_total_train_images": 20000,
        "actual_labeled_server_train": server_summary[0]["num_images"],
        "actual_labeled_server_val": server_summary[1]["num_images"],
        "actual_unlabeled_client_train": unlabeled_total,
        "actual_total_train_images": server_summary[0]["num_images"] + unlabeled_total,
        "materialization": "copied_images_verified_with_pil",
    },
    "bad_required_images_report": str((REPAIR_REPORT_ROOT / "bad_required_images.csv").resolve()),
    "num_bad_required_images_seen_before_selection_limit": len(bad_rows),
}
(PAPER_DATA_ROOT / "dataset_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "server_weather": "partly cloudy",
  "client_weathers": [
    "overcast",
    "rainy",
    "snowy"
  ],
  "server": [
    {
      "split": "train",
      "num_images": 4881,
      "num_boxes": 97123,
      "class_counts": {
        "traffic sign": 18325,
        "person": 6370,
        "car": 55752,
        "bus": 1112,
        "traffic light": 11415,
        "bike": 488,
        "truck": 2993,
        "motor": 270,
        "rider": 376,
        "train": 22
      }
    },
    {
      "split": "val",
      "num_images": 738,
      "num_boxes": 14937,
      "class_counts": {
        "car": 8622,
        "person": 1052,
        "traffic light": 1619,
        "traffic sign": 2845,
        "truck": 467,
        "bus": 151,
        "motor": 65,
        "bike": 70,
        "rider": 44,
        "train": 2
      }
    }
  ],
  "clients": [
    {
      "client_id": 0,
      "weather": "overcast",
      "num_images": 5000
    },
    {
      "client_id": 1,
      "weather": "rainy",
      "num

## 5. Regenerate EfficientTeacher Lists and Remove Stale Caches


In [6]:
subprocess.run([sys.executable, str(PROJECT_ROOT / "setup_fedsto_exact_reproduction.py")], cwd=PROJECT_ROOT, check=True)

removed = []
for root in [PAPER_DATA_ROOT, PROJECT_ROOT / "efficientteacher_fedsto" / "data_lists"]:
    for cache_path in root.rglob("*.cache"):
        cache_path.unlink()
        removed.append(str(cache_path))

print("removed cache files:", len(removed))
for path in removed[:20]:
    print("-", path)


{
  "manifest": {
    "paper": "Navigating Data Heterogeneity in Federated Learning: A Semi-Supervised Federated Object Detection",
    "official_ssfod_repo": "https://github.com/Kthyeon/ssfod",
    "official_ssfod_sha": "9e114950a7732b2367d7929fbf5409daa3d585b9",
    "efficientteacher_repo": "https://github.com/AlibabaResearch/efficientteacher",
    "efficientteacher_sha": "64165e24a30cd6dc1dd893cc861c1feab0a536ee",
    "server": {
      "weather": "cloudy represented by BDD100K Kaggle weather='partly cloudy'",
      "train_list": "/app/Object_Detection/navigating_data_heterogeneity/efficientteacher_fedsto/data_lists/server_cloudy_train.txt",
      "val_list": "/app/Object_Detection/navigating_data_heterogeneity/efficientteacher_fedsto/data_lists/server_cloudy_val.txt",
      "train_images": 4881,
      "val_images": 738
    },
    "clients": [
      {
        "id": 0,
        "weather": "overcast",
        "list": "/app/Object_Detection/navigating_data_heterogeneity/efficientteacher_

## 6. Verify Rebuilt Images


In [7]:
VERIFY_ALL_COPIED_IMAGES = True

image_dirs = [
    PAPER_DATA_ROOT / "server" / "images" / "train",
    PAPER_DATA_ROOT / "server" / "images" / "val",
    *(PAPER_DATA_ROOT / "clients" / f"client_{idx}" / "images_unlabeled" for idx in range(3)),
]

bad = []
total = 0
for directory in image_dirs:
    files = sorted(directory.glob("*.jpg"))
    sample = files if VERIFY_ALL_COPIED_IMAGES else files[:20]
    total += len(sample)
    for path in sample:
        ok, _size, reason = verify_image(path)
        if not ok:
            bad.append({"path": str(path), "reason": reason})

print("verified images:", total)
print("bad copied images:", len(bad))
if bad:
    write_csv(REPAIR_REPORT_ROOT / "bad_copied_images.csv", bad)
    raise RuntimeError(f"Bad copied images remain. See {REPAIR_REPORT_ROOT / 'bad_copied_images.csv'}")

print("data_paper20k is materialized and ready for 03_fedsto_exact_reproduction.ipynb")


verified images: 20619
bad copied images: 0
data_paper20k is materialized and ready for 03_fedsto_exact_reproduction.ipynb
